# CS 549 Project

## Table Construction

In [1]:
import numpy as np
import pandas as pd

In [5]:
mendely1 = pd.read_csv("../data/Phishing URLs.csv")
mendely1

,url,Type
0,https://docs.google.com/presentation/d/e/2PACX...,Phishing
1,https://btttelecommunniccatiion.weeblysite.com/,Phishing
2,https://kq0hgp.webwave.dev/,Phishing
3,https://brittishtele1bt-69836.getresponsesite....,Phishing
4,https://bt-internet-105056.weeblysite.com/,Phishing
...,...,...
54802,http://www.ezblox.site/free/jennifer111/helpdesk,Phishing
54803,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...,Phishing
54804,http://www.formbuddy.com/cgi-bin/formdisp.pl?u...,Phishing
54805,http://www.habbocreditosparati.blogspot.com/,Phishing


In [6]:
mendely2 = pd.read_csv("../data/urldata.csv")
mendely2

,url,type
0,https://www.google.com,legitimate
1,https://www.youtube.com,legitimate
2,https://www.facebook.com,legitimate
3,https://www.baidu.com,legitimate
4,https://www.wikipedia.org,legitimate
...,...,...
450171,http://ecct-it.com/docmmmnn/aptgd/index.php,phishing
450172,http://faboleena.com/js/infortis/jquery/plugin...,phishing
450173,http://faboleena.com/js/infortis/jquery/plugin...,phishing
450174,http://atualizapj.com/,phishing


In [7]:
kaggleDf = pd.read_csv("../data/malicious_phish.csv")
kaggleDf

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement
...,...,...
651186,xbox360.ign.com/objects/850/850402.html,phishing
651187,games.teamxbox.com/xbox-360/1860/Dead-Space/,phishing
651188,www.gamespot.com/xbox360/action/deadspace/,phishing
651189,en.wikipedia.org/wiki/Dead_Space_(video_game),phishing


In [8]:
mendely1 = mendely1.rename(columns={"Type": "type"})[['url', 'type']]

In [9]:
df = pd.concat([mendely1, mendely2, kaggleDf], ignore_index=True)
df = df.dropna(subset=['url', 'type'])
df = df.drop_duplicates(subset=['url']).reset_index(drop=True)

df = df.drop(columns=[c for c in ["Unnamed: 0",  "result"] if c in df.columns] )
df

,url,type
0,https://docs.google.com/presentation/d/e/2PACX...,Phishing
1,https://btttelecommunniccatiion.weeblysite.com/,Phishing
2,https://kq0hgp.webwave.dev/,Phishing
3,https://brittishtele1bt-69836.getresponsesite....,Phishing
4,https://bt-internet-105056.weeblysite.com/,Phishing
...,...,...
1144949,xbox360.ign.com/objects/850/850402.html,phishing
1144950,games.teamxbox.com/xbox-360/1860/Dead-Space/,phishing
1144951,www.gamespot.com/xbox360/action/deadspace/,phishing
1144952,en.wikipedia.org/wiki/Dead_Space_(video_game),phishing


In [10]:
# Standardize label text
df["type"] = df["type"].astype(str).str.strip().str.lower()

print("Raww label counts before class: ")
print(df["type"].value_counts(dropna=False))

# binary classification
label_map = {
    "legitimate": 0,
    "benign": 0,
    "safe": 0,             # in case this appears
    "phishing": 1,
    "malware": 1,
    "defacement": 1
}

# Apply class
df["label"] = df["type"].map(label_map)

# Drop non classed labels
before_rows = len(df)
df = df.dropna(subset=["label"]).reset_index(drop=True)
after_rows = len(df)

print(f"\n Dropped {before_rows - after_rows} rows with unmapped labels ")

#convert to int
df["label"] = df["label"].astype(int)

print("\nLabel #'s after cleaning: ")
print(df["label"].value_counts())

Raww label counts before class: 
type
benign        428080
legitimate    345738
phishing      252184
defacement     95308
malware        23644
Name: count, dtype: int64

 Dropped 0 rows with unmapped labels 

Label #'s after cleaning: 
label
0    773818
1    371136
Name: count, dtype: int64


In [11]:
df



,url,type,label
0,https://docs.google.com/presentation/d/e/2PACX...,phishing,1
1,https://btttelecommunniccatiion.weeblysite.com/,phishing,1
2,https://kq0hgp.webwave.dev/,phishing,1
3,https://brittishtele1bt-69836.getresponsesite....,phishing,1
4,https://bt-internet-105056.weeblysite.com/,phishing,1
...,...,...,...
1144949,xbox360.ign.com/objects/850/850402.html,phishing,1
1144950,games.teamxbox.com/xbox-360/1860/Dead-Space/,phishing,1
1144951,www.gamespot.com/xbox360/action/deadspace/,phishing,1
1144952,en.wikipedia.org/wiki/Dead_Space_(video_game),phishing,1


In [14]:
df['type'].unique()


array(['phishing', 'legitimate', 'benign', 'defacement', 'malware'],
      dtype=object)

## Feature Engineering

In [15]:
import math
import re
from urllib.parse import urlparse

from sklearn.feature_extraction.text import TfidfVectorizer

In [16]:
df

,url,type,label
0,https://docs.google.com/presentation/d/e/2PACX...,phishing,1
1,https://btttelecommunniccatiion.weeblysite.com/,phishing,1
2,https://kq0hgp.webwave.dev/,phishing,1
3,https://brittishtele1bt-69836.getresponsesite....,phishing,1
4,https://bt-internet-105056.weeblysite.com/,phishing,1
...,...,...,...
1144949,xbox360.ign.com/objects/850/850402.html,phishing,1
1144950,games.teamxbox.com/xbox-360/1860/Dead-Space/,phishing,1
1144951,www.gamespot.com/xbox360/action/deadspace/,phishing,1
1144952,en.wikipedia.org/wiki/Dead_Space_(video_game),phishing,1


### Keyword Extraction

In [17]:
KEYWORDS = [
    "pay", "ticket", "fine", "refund", "claim", "submit", "verify", "update",
    "account", "login", "secure", "reset", "password", "invoice", "shipping",
    "delivery", "package"
]

keyword_pattern = re.compile("|".join(KEYWORDS), re.IGNORECASE)

def keyword_count(url: str):
    matches = keyword_pattern.findall(url)
    return len(matches)

df["keyword_count"] = df["url"].apply(keyword_count)
df.head()

,url,type,label,keyword_count
0,https://docs.google.com/presentation/d/e/2PACX...,phishing,1,0
1,https://btttelecommunniccatiion.weeblysite.com/,phishing,1,0
2,https://kq0hgp.webwave.dev/,phishing,1,0
3,https://brittishtele1bt-69836.getresponsesite....,phishing,1,0
4,https://bt-internet-105056.weeblysite.com/,phishing,1,0


In [20]:
df['keyword_count'].unique()


array([ 0,  2,  1,  3,  4,  5,  7,  6, 12,  8, 16, 14, 15, 10, 13, 11, 18,
        9])

### More Lexical Features - URL length, digit count, special char, etc...

In [ ]:
import string
from urllib.parse import urlparse
import math


# Shannon entropy function
def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0
    from collections import Counter
    counts = Counter(s)
    total = len(s)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())



# Extract core lexical features
def extract_lexical_features(url: str):
    parsed = urlparse(url)

    host = parsed.netloc
    path = parsed.path + ("?" + parsed.query if parsed.query else "")

    # lengths
    url_len = len(url)
    host_len = len(host)
    path_len = len(path)



    # Char counts
    num_digits = sum(c.isdigit() for c in url)
    num_special = sum(c in string.punctuation for c in url)
    num_letters = sum(c.isalpha() for c in url)



    #URL structure counts
    num_dots = url.count('.')
    num_slashes = url.count('/')

    # shannon entropy for randomnes
    entropy_value = shannon_entropy(url)

    return pd.Series({
        "url_length": url_len,
        "host_length": host_len,
        "path_length": path_len,
        "num_digits": num_digits,
        "num_special": num_special,
        "num_letters": num_letters,
        "num_dots": num_dots,
        "num_slashes": num_slashes,
        "entropy": entropy_value
    })



# feature extractor
lexical_features = df["url"].apply(extract_lexical_features)

# Combines with the original dataframe
df_features = pd.concat([df, lexical_features], axis=1)

df_features.head()

,url,type,label,keyword_count,url_length,host_length,path_length,num_digits,num_special,num_letters,num_dots,num_slashes,entropy
0,https://docs.google.com/presentation/d/e/2PACX...,phishing,1,0,178.0,15.0,155.0,20.0,23.0,135.0,3.0,7.0,5.597944
1,https://btttelecommunniccatiion.weeblysite.com/,phishing,1,0,47.0,38.0,1.0,0.0,6.0,41.0,2.0,3.0,3.974149
2,https://kq0hgp.webwave.dev/,phishing,1,0,27.0,18.0,1.0,1.0,6.0,20.0,2.0,3.0,3.958229
3,https://brittishtele1bt-69836.getresponsesite....,phishing,1,0,50.0,41.0,1.0,6.0,7.0,37.0,2.0,3.0,4.151272
4,https://bt-internet-105056.weeblysite.com/,phishing,1,0,42.0,33.0,1.0,6.0,8.0,28.0,2.0,3.0,4.252453


In [31]:
df_features

,url,type,label,keyword_count,url_length,host_length,path_length,num_digits,num_special,num_letters,num_dots,num_slashes,entropy
0,https://docs.google.com/presentation/d/e/2PACX...,phishing,1,0,178.0,15.0,155.0,20.0,23.0,135.0,3.0,7.0,5.597944
1,https://btttelecommunniccatiion.weeblysite.com/,phishing,1,0,47.0,38.0,1.0,0.0,6.0,41.0,2.0,3.0,3.974149
2,https://kq0hgp.webwave.dev/,phishing,1,0,27.0,18.0,1.0,1.0,6.0,20.0,2.0,3.0,3.958229
3,https://brittishtele1bt-69836.getresponsesite....,phishing,1,0,50.0,41.0,1.0,6.0,7.0,37.0,2.0,3.0,4.151272
4,https://bt-internet-105056.weeblysite.com/,phishing,1,0,42.0,33.0,1.0,6.0,8.0,28.0,2.0,3.0,4.252453
...,...,...,...,...,...,...,...,...,...,...,...,...,...
695854,xbox360.ign.com/objects/850/850402.html,phishing,1,0,39.0,0.0,39.0,12.0,6.0,21.0,3.0,3.0,4.355539
695855,games.teamxbox.com/xbox-360/1860/Dead-Space/,phishing,1,0,44.0,0.0,44.0,7.0,8.0,29.0,2.0,4.0,4.243300
695856,www.gamespot.com/xbox360/action/deadspace/,phishing,1,0,42.0,0.0,42.0,3.0,6.0,33.0,2.0,4.0,4.147921
695857,en.wikipedia.org/wiki/Dead_Space_(video_game),phishing,1,0,45.0,0.0,45.0,0.0,9.0,36.0,2.0,2.0,4.102313


In [1]:
import pandas as pd

df = pd.read_csv("../csv/final_dataset.csv")

df[[
    "digit_proportion",
    "letter_proportion",
    "special_proportion"
]].head()

df[[
    "digit_proportion",
    "letter_proportion",
    "special_proportion"
]].describe()

,digit_proportion,letter_proportion,special_proportion
count,1.040564e+06,1.040564e+06,1.040564e+06
mean,6.422389e-02,7.726685e-01,1.630688e-01
std,9.044114e-02,1.019865e-01,4.341400e-02
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,7.323944e-01,1.348315e-01
50%,2.702703e-02,7.916667e-01,1.641791e-01
75%,9.803922e-02,8.378378e-01,1.891892e-01
max,8.000000e-01,1.000000e+00,8.119658e-01
